In [2]:
import os

os.getcwd()

'e:\\chatbot\\booking'

In [3]:
# sqlite connection

import sqlite3

conn=sqlite3.connect(r'e:\chatbot\\booking\hospital.db')
cursor=conn.cursor()

In [4]:
# will create three tables

cursor.execute("""
CREATE TABLE IF NOT EXISTS doctors (
    doctor_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    morning_start TEXT NOT NULL,
    morning_end TEXT NOT NULL,
    break_start TEXT NOT NULL,
    break_end TEXT NOT NULL,
    evening_start TEXT NOT NULL,
    evening_end TEXT NOT NULL
)
""")

conn.commit()

In [5]:
cursor.execute("PRAGMA table_info(doctors)")
cursor.fetchall()

[(0, 'doctor_id', 'INTEGER', 0, None, 1),
 (1, 'name', 'TEXT', 1, None, 0),
 (2, 'morning_start', 'TEXT', 1, None, 0),
 (3, 'morning_end', 'TEXT', 1, None, 0),
 (4, 'break_start', 'TEXT', 1, None, 0),
 (5, 'break_end', 'TEXT', 1, None, 0),
 (6, 'evening_start', 'TEXT', 1, None, 0),
 (7, 'evening_end', 'TEXT', 1, None, 0)]

In [7]:
doctors = [
    ("Dr. Sharma", "11:00", "13:00", "13:00", "14:00", "14:00", "20:00"),
    ("Dr. Khan",   "10:00", "13:00", "13:00", "14:00", "14:00", "19:00")
]

cursor.executemany("""
    INSERT INTO doctors ( name, morning_start, morning_end,
    break_start, break_end,
    evening_start, evening_end
) VALUES
 (?, ?, ?, ?, ?, ?, ?)
""", doctors)
conn.commit()

In [8]:
cursor.execute('select * from doctors')
cursor.fetchall()

[(1, 'Dr. Sharma', '11:00', '13:00', '13:00', '14:00', '14:00', '20:00'),
 (2, 'Dr. Khan', '10:00', '13:00', '13:00', '14:00', '14:00', '19:00')]

In [11]:
cursor.execute("select * from doctors where name='Dr. Sharma'")
res=cursor.fetchone()
if res:
    print("dr is in the list")
else:
    print("this dr isnt mentioned in our db")

dr is in the list


In [12]:
cursor.execute("select * from doctors where name='Dr. Sharma'")
res=cursor.fetchone()

In [13]:
res

(1, 'Dr. Sharma', '11:00', '13:00', '13:00', '14:00', '14:00', '20:00')

In [14]:
res[0]

1

In [15]:
res[1]

'Dr. Sharma'

In [16]:
res[2]

'11:00'

In [17]:
res

(1, 'Dr. Sharma', '11:00', '13:00', '13:00', '14:00', '14:00', '20:00')

In [24]:
def check_availability(dr_name,session):
    cursor.execute('select * from doctors where name= ?',(dr_name,))
    doctor=cursor.fetchone()

    if doctor is None:
        return (f"sorry {dr_name} was not found!")

    #get all the details from doctor variable which is a tupele we can acces the values from indexes
    id=doctor[0]
    name=doctor[1]
    morning_start=doctor[2]
    morning_end=doctor[3]
    break_start=doctor[4]
    break_end=doctor[5]
    evening_start = doctor[6]
    evening_end = doctor[7]

    session=session.lower().strip()

    if session=='morning':
        return (
            f"{name} is available in the morning "
            f"from {morning_start} to {morning_end}."
        )

    elif session == "evening":
        return (
            f"{name} is available in the evening "
            f"from {evening_start} to {evening_end}."
        )

    else:
        return "Please choose either morning or evening."

In [25]:
res=check_availability('Dr. Sharma','morning')
print(res)

Dr. Sharma is available in the morning from 11:00 to 13:00.


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS appointments (
    appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_name TEXT NOT NULL,
    phone TEXT NOT NULL,
    doctor_name TEXT NOT NULL,
    appointment_date TEXT NOT NULL,
    session TEXT NOT NULL,
    queue_number INTEGER NOT NULL
)
""")

conn.commit()

print("Appointments table created successfully!")

Appointments table created successfully!


In [33]:
def book_appointment(
    patient_name: str,
    phone: str,
    dr_name: str,
    appointment_date: str,
    session: str
):
    session = session.strip().lower()

    if session not in ("morning", "evening"):
        return "Please choose morning or evening."

    cursor.execute('select * from doctors where name= ?',(dr_name,))
    doctor=cursor.fetchone()
    
    if doctor is None:
        return (f"sorry {dr_name} was not found!")


    cursor.execute("""
        SELECT COALESCE(MAX(queue_number),0)+1
        FROM appointments
        WHERE LOWER(doctor_name) = Lower(?) and
        appointment_date =? and
        session =? 
    """,(dr_name,appointment_date,session))

    queue_number=cursor.fetchone()[0]

    cursor.execute("""
        INSERT INTO appointments (
            patient_name, phone, doctor_name,
            appointment_date, session, queue_number
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        patient_name, phone, dr_name,
        appointment_date, session, queue_number
    ))

    conn.commit()

    return (
        f"Appointment booked successfully! "
        f"Queue number: {queue_number}. "
        f"Doctor: {dr_name}. "
        f"Date: {appointment_date}. "
        f"Session: {session}. "
        f"Your exact consultation time is not guaranteed."
    )

In [34]:
# def book_appointment(
#     patient_name: str,
#     phone: str,
#     dr_name: str,
#     appointment_date: str,
#     session: str
# ):

book_appointment('Adeena Mahaskar','9876543211','Dr. Sharma','10/31/2026','morning')

'Appointment booked successfully! Queue number: 1. Doctor: Dr. Sharma. Date: 10/31/2026. Session: morning. Your exact consultation time is not guaranteed.'

In [35]:
book_appointment('Sana Shaikh','9876543212','Dr. Sharma','10/31/2026','morning')

'Appointment booked successfully! Queue number: 2. Doctor: Dr. Sharma. Date: 10/31/2026. Session: morning. Your exact consultation time is not guaranteed.'

In [36]:
cursor

In [41]:
# Close the global connection, if it exists
try:
    conn.close()
except NameError:
    pass